# R3.09 TP 5 : Problème du logarithme discret — Chiffre d'El Gamal

**Étudiant :** Mamadou-Bailo BARRY - BUT Informatique 2 (Site d'Arles)
**Enseignant :** M. Bruno Colombel
**Environnement :** Python 3 (Jupyter)

## 1 Problème du logarithme discret

### 1.1 Fonction indicatrice d'Euler (rappels)

- $a$ est inversible modulo $n$ si, et seulement si, il existe $b \in \mathbb{Z}/n\mathbb{Z}$ tel que $a \times b \equiv 1 \pmod n$.
- Si $p$ est premier, tout élément non nul de $\mathbb{Z}/p\mathbb{Z}$ est inversible et $\varphi(p) = p - 1$.
- **Théorème 2 :** si $a$ et $n$ sont premiers entre eux, alors $a^{\varphi(n)} \equiv 1 \pmod n$, d'où :
$$a^{-1} \equiv a^{\varphi(n) - 1} \pmod n$$

Avec $p$ premier, on aura donc $a^{-1} \equiv a^{p-2} \pmod p$ (on s'en servira pour le déchiffrement d'El Gamal).

### 1.2 Racine primitive — logarithme discret

On reprend la fonction `puissance(a, x, n)` donnée dans le sujet (algorithme des carrés, version récursive) :

In [ ]:
def puissance(a, x, n):
    """methode récursive, rapide de calcul de a^x mod n
    Attention au cas x == 0"""
    if x == 0:
        return 1
    elif x == 1:
        return a
    elif x % 2 == 0:
        return puissance(a**2 % n, x // 2, n) % n
    elif x != 1:
        return a * puissance(a**2 % n, (x-1) // 2, n) % n

Et la fonction `cycle(a, p)` du sujet, qui retourne la liste des puissances $a^n \pmod p$ :

In [ ]:
def cycle(a, p):
    """retourne a**0, a**1, a**2, ...a**(p-1) modulo p """
    resultat = []
    for d in range(p-1):
        resultat.append(puissance(a, d, p))
    return resultat

### Exercice 1

1. On prend $p = 31$ et on affiche `cycle(a, p)` pour $a = 2, 3, 4$ et $5$ :

In [ ]:
p = 31
for a in [2, 3, 4, 5]:
    print("a =", a, ":", cycle(a, p))

2. Pour $a = 2$, les puissances reviennent à $1$ dès $2^5 = 32 \equiv 1 \pmod{31}$ : la liste répète le motif $1, 2, 4, 8, 16$.  
   La fonction $x \mapsto 2^x \pmod{31}$ ne prend que $5$ valeurs : deux $x$ différents donnent la même image (par exemple $2^0 \equiv 2^5 \equiv 1$), elle n'est donc **pas bijective**.  
   On observe le même comportement pour :
   - $a = 4$ : motif $1, 4, 16, 2, 8$ (5 valeurs) car $4^5 \equiv 1 \pmod{31}$ ;
   - $a = 5$ : motif $1, 5, 25$ (3 valeurs) car $5^3 = 125 = 4 \times 31 + 1 \equiv 1 \pmod{31}$.

On le vérifie avec `puissance` : $2^0$ et $2^5$ ont la même image, et $4^5$, $5^3$ valent aussi $1$ :

In [ ]:
print(puissance(2, 0, p), puissance(2, 5, p))
print(puissance(4, 5, p), puissance(5, 3, p))

3. Parmi les valeurs testées, seule **$a = 3$** rend la fonction $x \mapsto 3^x \pmod{31}$ bijective : dans sa liste, on trouve les $30$ éléments non nuls de $\mathbb{Z}/31\mathbb{Z}$ (de $1$ à $30$), chacun une seule fois. On réaffiche son cycle :

In [ ]:
print(cycle(3, p))

### Exercice 2

D'après la **Définition 3**, avec $p$ premier ($\varphi(p) = p - 1$), $a$ est une racine primitive de $p$ si pour tout $d$ avec $1 \leqslant d < p - 1$, $a^d \not\equiv 1 \pmod p$.

La première condition de la définition ($a$ premier avec $p$) est toujours vraie ici : $p$ est premier et $1 < a < p$ (**Théorème 1**). Il reste donc seulement à tester les puissances.

On calcule les puissances $a^d$ de proche en proche ($a^d = a^{d-1} \times a$) et on s'arrête dès qu'on tombe sur $1$.

In [ ]:
def primitive(a, p):
    """teste si a est une racine primitive de p avec p premier"""
    x = 1
    for d in range(1, p-1):
        x = x * a % p
        if x == 1:
            return False
    return True

In [ ]:
primitive(3, 31)

In [ ]:
primitive(4, 31)

### Exercice 3

1. On cherche une racine primitive en testant $a = 2, 3, 4, \dots$ jusqu'à en trouver une :

In [ ]:
def racine_primitive(p):
    """recherche une racine primitive de p avec p premier"""
    a = 2
    while not primitive(a, p):
        a = a + 1
    return a

In [ ]:
racine_primitive(4999)

In [ ]:
racine_primitive(1000021)

> **Remarque.** $1000021 = 11 \times 90911$ n'est pas premier. Comme $11$ n'est pas premier avec $1000021$, ses puissances ne valent jamais $1$ et `primitive(11, 1000021)` renvoie `True`. On retrouve bien le résultat `11` de l'énoncé, mais nos fonctions ne sont valables que pour $p$ premier.

2. Pour la liste des racines primitives, on teste tous les $a$ avec $1 < a < p$ :

In [ ]:
def liste_racines_primitives(p):
    """retourne la liste des racines primitives de p avec p premier"""
    resultat = []
    for a in range(2, p):
        if primitive(a, p):
            resultat.append(a)
    return resultat

In [ ]:
liste_racines_primitives(31)

On retrouve les valeurs de l'énoncé : $3, 11, 12, 13, 17, 21, 22$ et $24$.

### 1.3 Logarithme discret

**Exemple 1.** $3$ est une racine primitive de $31$ et $\log_3(19) = 4 \pmod{31}$ car $3^4 = 81 = 2 \times 31 + 19$. On le vérifie :

In [ ]:
puissance(3, 4, 31)

## 2 Chiffre d'El Gamal

Rappel du sujet :

- **Génération des clés (Alice) :** $p$ premier, $g$ racine primitive de $p$, $a$ choisi au hasard et $A = g^a \pmod p$.
  Clé publique : $(p, g, A)$ — clé privée : $a$.
- **Chiffrement (Bob) :** $b$ choisi au hasard entre $0$ et $p - 2$, $c = m A^b$ et $B = g^b$. Bob envoie $(c, B)$.
- **Déchiffrement (Alice) :** $m \equiv c \times (B^a)^{-1} \pmod p$, car $(B^a)^{-1} = (g^{ba})^{-1}$ et $A^b = g^{ab}$.

### Exercice 4

On suit les étapes proposées : `randprime(2**k, 2**(k+1))` (bibliothèque `sympy`) renvoie un nombre premier $p$ avec $2^k \leqslant p < 2^{k+1}$, puis on choisit la racine primitive $g$ et la clé privée au hasard avec `randint` (bibliothèque `random`).

In [ ]:
from random import randint
from sympy import randprime

def clefs(k):
    """génère une paire de clefs El Gamal avec un module p entre 2**k et 2**(k+1) - 1"""
    p = randprime(2**k, 2**(k+1))
    g = racine_primitive(p)
    a = randint(0, p-2)
    A = puissance(g, a, p)
    public_key = (p, g, A)
    private_key = a
    return public_key, private_key

In [ ]:
k = 16
public_key, private_key = clefs(k)
print(public_key, private_key)

### Exercice 5

Bob choisit $b$ au hasard entre $0$ et $p - 2$, puis calcule $c = m A^b$ et $B = g^b \pmod p$.

Comme dans l'exemple du sujet, $c$ n'est pas réduit modulo $p$ (c'est pour ça que $c$ peut être plus grand que $p$) : la réduction est faite au déchiffrement.

In [ ]:
def chiffrer(m, public_key):
    """chiffre le nombre m (0 <= m <= p-1) avec la clé publique (p, g, A)"""
    (p, g, A) = public_key
    b = randint(0, p-2)
    c = m * puissance(A, b, p)
    B = puissance(g, b, p)
    return c, B

In [ ]:
(p, g, A) = public_key
m = randint(0, p-1)

c, B = chiffrer(m, public_key)
print(c, B)

### Exercice 6

Alice calcule $c \times (B^a)^{-1} \pmod p$.
D'après le **Théorème 2** avec $p$ premier : $(B^a)^{-1} \equiv (B^a)^{p-2} \pmod p$.

**Remarque.** Le calcul se fait modulo $p$, qui n'est pas dans la clé privée $a$ : on passe donc aussi la clé publique à la fonction (l'en-tête de l'énoncé `dechiffrer(c, B, private_key, private_key)` contient deux fois `private_key`, le deuxième paramètre est la clé publique).

In [ ]:
def dechiffrer(c, B, private_key, public_key):
    """déchiffre le message chiffré (c, B) avec la clé privée a"""
    (p, g, A) = public_key
    a = private_key
    Ba = puissance(B, a, p)
    inverse = puissance(Ba, p-2, p)
    return c * inverse % p

In [ ]:
res = dechiffrer(c, B, private_key, public_key)
res == m

### Exercice 7

Procédé du sujet : chaque caractère est remplacé par son code ASCII `ord(c)`, écrit sur 3 chiffres (on ajoute des 0 à gauche), puis on concatène tout et on transforme en `int`.

1. Fonction `encode(texte)` :

In [ ]:
def encode(texte):
    """transforme le texte en un nombre entier"""
    chaine = ""
    for c in texte:
        code = str(ord(c))
        while len(code) < 3:
            code = "0" + code
        chaine = chaine + code
    return int(chaine)

In [ ]:
encode("Bob")

On retrouve bien $B \to 066$, $o \to 111$, $b \to 098$ : `066111098` donne l'entier $66111098$ (le $0$ de gauche disparaît, ce n'est pas gênant).

2. Fonction `decode(nombre)` : avec l'astuce de l'énoncé, `nombre % 1000` donne le code ASCII du **dernier** caractère et `nombre // 1000` enlève ce caractère. On reconstruit donc le texte de droite à gauche avec `chr` :

In [ ]:
def decode(nombre):
    """retrouve le texte représenté par le nombre entier"""
    texte = ""
    while nombre > 0:
        num = nombre % 1000
        texte = chr(num) + texte
        nombre = nombre // 1000
    return texte

In [ ]:
decode(66111098)

3. Tests : pour tout message `m`, on doit avoir `decode(encode(m)) == m`.

In [ ]:
for message in ["Bob", "Alice", "Chiffre d'El Gamal", "R3.09 - TP5"]:
    print(message, ":", decode(encode(message)) == message)

### Exercice 8

Le nombre qui encode le message doit être plus petit que le module $p$. Avec $k = 20$, on a $p \geqslant 2^{20} = 1\,048\,576$ : un message de **2 caractères** (6 chiffres, donc au plus $999\,999$) passe toujours.

**Ma paire de clés** (je donne `ma_cle_publique` à mon voisin, je garde `ma_cle_privee`) :

In [ ]:
ma_cle_publique, ma_cle_privee = clefs(20)
print("clé publique à donner au voisin :", ma_cle_publique)

**J'envoie un message à mon voisin** : on remplace la clé ci-dessous par la clé publique qu'il nous communique (ici on en génère une pour la démonstration).

In [ ]:
cle_publique_voisin, cle_privee_voisin = clefs(20)   # à remplacer par la clé publique du voisin

message = "OK"
m = encode(message)
print("message encodé :", m, "< p :", m < cle_publique_voisin[0])

c, B = chiffrer(m, cle_publique_voisin)
print("cryptogramme envoyé (c, B) :", c, B)

**Le voisin déchiffre** avec sa clé privée, puis décode le nombre obtenu :

In [ ]:
nombre = dechiffrer(c, B, cle_privee_voisin, cle_publique_voisin)
decode(nombre)

**Je reçois un cryptogramme $(c, B)$ de mon voisin** (chiffré avec `ma_cle_publique`) : je le déchiffre avec ma clé privée. Ici on simule l'envoi du voisin.

In [ ]:
c, B = chiffrer(encode("Ca"), ma_cle_publique)   # à remplacer par le (c, B) reçu du voisin

decode(dechiffrer(c, B, ma_cle_privee, ma_cle_publique))

### Exercice 9 (Bonus)

Pour un message long, on découpe l'entier en morceaux plus petits que $p$, avec la même astuce que pour `decode` mais en divisant par $1000^{\text{taille}}$ au lieu de $1000$ : chaque morceau contient `taille` caractères (3 chiffres par caractère), ce qui évite de couper un code ASCII en deux.

On prend `taille` le plus grand possible avec $1000^{\text{taille}} \leqslant p$ : c'est le nombre de chiffres de $p$ moins 1, divisé par 3.

In [ ]:
def decouper(nombre, taille):
    """découpe le nombre en blocs de taille caractères (3*taille chiffres)"""
    blocs = []
    while nombre > 0:
        blocs = [nombre % 1000**taille] + blocs
        nombre = nombre // 1000**taille
    return blocs

def recoller(blocs, taille):
    """opération réciproque de decouper"""
    nombre = 0
    for bloc in blocs:
        nombre = nombre * 1000**taille + bloc
    return nombre

In [ ]:
def chiffrer_texte(texte, public_key):
    """chiffre un texte de longueur quelconque, bloc par bloc"""
    (p, g, A) = public_key
    taille = (len(str(p)) - 1) // 3
    resultat = []
    for bloc in decouper(encode(texte), taille):
        resultat.append(chiffrer(bloc, public_key))
    return resultat

def dechiffrer_texte(cryptogramme, private_key, public_key):
    """déchiffre la liste des couples (c, B) et retrouve le texte"""
    (p, g, A) = public_key
    taille = (len(str(p)) - 1) // 3
    blocs = []
    for (c, B) in cryptogramme:
        blocs.append(dechiffrer(c, B, private_key, public_key))
    return decode(recoller(blocs, taille))

Test avec une clé de $k = 20$ (2 caractères par bloc) :

In [ ]:
texte = "Le chiffre d'El Gamal repose sur le probleme du logarithme discret."

cryptogramme = chiffrer_texte(texte, ma_cle_publique)
print("nombre de blocs :", len(cryptogramme))

dechiffrer_texte(cryptogramme, ma_cle_privee, ma_cle_publique) == texte